In [1]:
!apt-get -qq install -y ffmpeg
!pip -q install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 44.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

CUDA available: False
Device: none


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import zipfile
from pathlib import Path

DRIVE_ZIP_PATH = "/content/drive/MyDrive/data/Interview-Task.zip"  # <- update

with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    zf.extractall("/content/Androids-Corpus")

print("Extracted to /content/Androids-Corpus")

Extracted to /content/Androids-Corpus


In [5]:
INPUT_DIR = Path("/content/Androids-Corpus/Interview-Task/audio")
for label_folder in ["HC", "PT"]:
    n_files = len(list((INPUT_DIR / label_folder).glob("*.wav")))
    print(f"{label_folder}: {n_files} files")

HC: 52 files
PT: 64 files


In [6]:
OUTPUT_DIR = Path("/content/drive/MyDrive/extracted_transcripts/transcripts")  # <- update, same parent as the zip
for label_folder in ["HC", "PT"]:
    (OUTPUT_DIR / label_folder).mkdir(parents=True, exist_ok=True)
print("Output folders ready:", list(OUTPUT_DIR.glob("*")))

Output folders ready: [PosixPath('/content/drive/MyDrive/extracted_transcripts/transcripts/HC'), PosixPath('/content/drive/MyDrive/extracted_transcripts/transcripts/PT')]


In [7]:
import json
import whisper

AUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".flac"}

def find_audio_files(input_dir: Path):
    return sorted(p for p in input_dir.rglob("*") if p.suffix.lower() in AUDIO_EXTENSIONS)

def relative_output_paths(audio_path: Path, input_dir: Path, output_dir: Path):
    rel_dir = audio_path.parent.relative_to(input_dir)
    stem = audio_path.stem
    return output_dir / rel_dir / f"{stem}.txt", output_dir / rel_dir / f"{stem}.json"

def transcribe_file(model, audio_path, txt_path, json_path, language="it"):
    result = model.transcribe(str(audio_path), language=language, task="transcribe", verbose=False)
    txt_path.write_text(result["text"].strip(), encoding="utf-8")
    json_path.write_text(
        json.dumps({
            "audio_file": audio_path.name,
            "language": result.get("language", language),
            "text": result["text"].strip(),
            "segments": [{"start": s["start"], "end": s["end"], "text": s["text"].strip()} for s in result["segments"]],
        }, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return len(result["text"].split())

In [ ]:
model = whisper.load_model("large-v3", device="cuda")

audio_files = find_audio_files(INPUT_DIR)
print(f"Found {len(audio_files)} files")

for i, audio_path in enumerate(audio_files, 1):
    txt_path, json_path = relative_output_paths(audio_path, INPUT_DIR, OUTPUT_DIR)
    if txt_path.exists():
        continue
    print(f"[{i}/{len(audio_files)}] {audio_path.relative_to(INPUT_DIR)}")
    try:
        n_words = transcribe_file(model, audio_path, txt_path, json_path)
        print(f"    -> {n_words} words")
    except Exception as e:
        print(f"    !! FAILED: {e}")

100%|██████████████████████████████████████| 2.88G/2.88G [00:15<00:00, 195MiB/s]


Found 116 files
[40/116] HC/44_CF37_3.wav


100%|██████████| 26302/26302 [01:29<00:00, 292.67frames/s]


    -> 581 words
[41/116] HC/45_CF40_3.wav


100%|██████████| 27888/27888 [01:19<00:00, 351.55frames/s]


...

100%|██████████| 15409/15409 [00:25<00:00, 597.73frames/s]


    -> 281 words
[96/116] PT/50_PF35_3.wav


100%|██████████| 12862/12862 [00:41<00:00, 311.90frames/s]


    -> 277 words
[97/116] PT/51_PF58_2.wav


 49%|████▉     | 5724/11582 [00:09<00:09, 592.22frames/s]